# Convert Pytorch weight to be tiny-engine compatible

In [1]:
import os
os.environ['https_proxy'] = 'http://192.168.1.22:7890'
debug = True
if debug:
    # improve torch tensor printing
    import torch
    def custom_repr(self):
        return f'{{Tensor:{tuple(self.shape)}}} {original_repr(self)}'
    original_repr = torch.Tensor.__repr__
    torch.Tensor.__repr__ = custom_repr

In [2]:
# Load model directly
model_name = "Qwen/Qwen3-8B"
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    trust_remote_code=True, 
    torch_dtype=torch.float32,
    device_map="cpu"
)

/root/workspace/development/Qwen3.Ink.Cpp/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 5/5 [00:03<00:00,  1.50it/s]


In [3]:
model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 4096)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (down_proj): Linear(in_features=12288, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
        (post_attention_layernorm): 

In [4]:
model.model

Qwen3Model(
  (embed_tokens): Embedding(151936, 4096)
  (layers): ModuleList(
    (0-35): 36 x Qwen3DecoderLayer(
      (self_attn): Qwen3Attention(
        (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
        (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
        (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
        (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
        (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
      )
      (mlp): Qwen3MLP(
        (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
        (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
        (down_proj): Linear(in_features=12288, out_features=4096, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
      (post_attention_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
    )
  )
  (norm): Qwen3RMSNorm((

In [5]:
model.model.layers[0].input_layernorm.weight

Parameter containing:
{Tensor:(4096,)} tensor([0.0111, 0.0107, 0.0126,  ..., 0.0124, 0.0113, 0.0115],
       requires_grad=True)

In [6]:
outdir_root = "../model_weights/int4/qwen3-8b-A81W41"
os.makedirs(outdir_root, exist_ok=True)

In [7]:
import numpy as np
import importlib
import quantize_methods
from typing import Dict, Tuple, List
importlib.reload(quantize_methods)
quantize_method = quantize_methods.quantize_row_q41_tinyml

if_store_fp16 = False


def _write_weight_to_file(output_dir: str, qs: np.ndarray, d: np.ndarray, m: np.ndarray, zp: np.ndarray) -> None:
    assert qs.dtype == np.uint8
    assert d.dtype == np.float32
    assert m.dtype == np.float32
    assert zp.dtype == np.int8

    output_dict = {
        os.path.join(output_dir, "weight_int4.bin") : qs.tobytes(),
        os.path.join(output_dir, "scaling_factor_int4.bin"): d.tobytes(),
        os.path.join(output_dir, "offset_int4.bin"): m.tobytes(),
        os.path.join(output_dir, "zero_point_int4.bin"): zp.tobytes(),
    }
    
    for path, data in output_dict.items():
        with open(path, "wb") as f:
            f.write(data)
            print(f"Saved {path}")

def quantize_and_save(state_dict: Dict) -> None:
    for output_dir, weight in state_dict.items():
        os.makedirs(output_dir, exist_ok=True)
        with torch.no_grad():
            weight = weight.cpu().float()
            qs, d, m, zp = quantize_method(weight, STORE_FP16=if_store_fp16)
            _write_weight_to_file(output_dir, qs, d, m, zp)

## Save lm_head

In [8]:
model.lm_head

Linear(in_features=4096, out_features=151936, bias=False)

In [9]:
quanzation_state_dict = {
    os.path.join(outdir_root, "lm_head"): model.lm_head.weight
}
quantize_and_save(quanzation_state_dict)

Max error: 0.012561, mean error: 0.000000, relative error: 117.783569%
Saved ../model_weights/int4/qwen3-8b-A81W41/lm_head/weight_int4.bin
Saved ../model_weights/int4/qwen3-8b-A81W41/lm_head/scaling_factor_int4.bin
Saved ../model_weights/int4/qwen3-8b-A81W41/lm_head/offset_int4.bin
Saved ../model_weights/int4/qwen3-8b-A81W41/lm_head/zero_point_int4.bin


## Save embedding, final norm

In [10]:
embed_tokens = model.model.embed_tokens

In [11]:
final_norm = model.model.norm

In [12]:
fp32_state_dict = dict()
output_dir = os.path.join(outdir_root, "model")

# export wte
fp32_state_dict.update(
    embed_token = ( 
        embed_tokens.weight, os.path.join(output_dir, "embed_tokens", "weight.bin") 
    ),
    final_norm = 
    (
        final_norm.weight, os.path.join(output_dir, "norm", "weight.bin")
    )
)


for _, path in fp32_state_dict.values():
    dirname = os.path.dirname(path)
    os.makedirs(dirname, exist_ok=True)

# print(OUTPUT_PATH)
with torch.no_grad():
    for name, (data, path) in fp32_state_dict.items():
        with open(path, "wb") as f:
            f.write(data.cpu().float().numpy().tobytes())
            print(f"\033[92mSaved {name} to {path}\033[0m")  # Green colored output
            


Saved embed_token to ../model_weights/int4/qwen3-8b-A81W41/model/embed_tokens/weight.bin
Saved final_norm to ../model_weights/int4/qwen3-8b-A81W41/model/norm/weight.bin


## Save Pre-computed rotary embedding with maximum sequence length

In [13]:
rotary_emb = model.model.rotary_emb


In [14]:
rotary_emb.attention_scaling

1.0

In [15]:
rotary_emb.inv_freq


{Tensor:(64,)} tensor([1.0000e+00, 8.0584e-01, 6.4938e-01, 5.2330e-01, 4.2170e-01, 3.3982e-01,
        2.7384e-01, 2.2067e-01, 1.7783e-01, 1.4330e-01, 1.1548e-01, 9.3057e-02,
        7.4989e-02, 6.0430e-02, 4.8697e-02, 3.9242e-02, 3.1623e-02, 2.5483e-02,
        2.0535e-02, 1.6548e-02, 1.3335e-02, 1.0746e-02, 8.6596e-03, 6.9783e-03,
        5.6234e-03, 4.5316e-03, 3.6517e-03, 2.9427e-03, 2.3714e-03, 1.9110e-03,
        1.5399e-03, 1.2409e-03, 1.0000e-03, 8.0584e-04, 6.4938e-04, 5.2330e-04,
        4.2170e-04, 3.3982e-04, 2.7384e-04, 2.2067e-04, 1.7783e-04, 1.4330e-04,
        1.1548e-04, 9.3057e-05, 7.4989e-05, 6.0430e-05, 4.8697e-05, 3.9242e-05,
        3.1623e-05, 2.5483e-05, 2.0535e-05, 1.6548e-05, 1.3335e-05, 1.0746e-05,
        8.6596e-06, 6.9783e-06, 5.6234e-06, 4.5316e-06, 3.6517e-06, 2.9427e-06,
        2.3714e-06, 1.9110e-06, 1.5399e-06, 1.2409e-06])

In [16]:
max_seq_length = 4096
device = model.device
position_ids = torch.arange(max_seq_length, device=device).reshape(1, max_seq_length)
# only provided device info
pseudo_x = torch.randn(1, max_seq_length, 4096, device=device)
cos, sin = rotary_emb(pseudo_x, position_ids)

In [17]:
cos

{Tensor:(1, 4096, 128)} tensor([[[ 1.0000,  1.0000,  1.0000,  ...,  1.0000,  1.0000,  1.0000],
         [ 0.5403,  0.6925,  0.7965,  ...,  1.0000,  1.0000,  1.0000],
         [-0.4161, -0.0409,  0.2687,  ...,  1.0000,  1.0000,  1.0000],
         ...,
         [-0.8799,  0.9359,  0.9914,  ...,  1.0000,  1.0000,  1.0000],
         [-0.8753,  0.9022,  0.7102,  ...,  1.0000,  1.0000,  1.0000],
         [-0.0660,  0.3139,  0.1399,  ...,  1.0000,  1.0000,  1.0000]]])

In [18]:
fp32_state_dict = dict()
# NOTE: All attention layers share the same pre-computed rope rotation matrix
output_dir = os.path.join(outdir_root, "model")

# export wte
fp32_state_dict.update(
    sin_cache = (sin, os.path.join(output_dir, "rotary_emb", "sin_cached.bin") ),
    cos_cache = (cos, os.path.join(output_dir, "rotary_emb", "cos_cached.bin") )
)
for _, path in fp32_state_dict.values():
    dirname = os.path.dirname(path)
    os.makedirs(dirname, exist_ok=True)

print(fp32_state_dict)
with torch.no_grad():
    for name, (data, path) in fp32_state_dict.items():
        with open(path, "wb") as f:
            f.write(data.cpu().float().numpy().tobytes())
            print(f"\033[92mSaved {name} to {path}\033[0m")  # Green colored output
    

{'sin_cache': ({Tensor:(1, 4096, 128)} tensor([[[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
           0.0000e+00,  0.0000e+00],
         [ 8.4147e-01,  7.2141e-01,  6.0469e-01,  ...,  1.9110e-06,
           1.5399e-06,  1.2409e-06],
         [ 9.0930e-01,  9.9916e-01,  9.6323e-01,  ...,  3.8219e-06,
           3.0799e-06,  2.4819e-06],
         ...,
         [ 4.7523e-01, -3.5230e-01,  1.3118e-01,  ...,  7.8215e-03,
           6.3029e-03,  5.0791e-03],
         [-4.8361e-01,  4.3125e-01,  7.0397e-01,  ...,  7.8234e-03,
           6.3044e-03,  5.0804e-03],
         [-9.9782e-01,  9.4947e-01,  9.9016e-01,  ...,  7.8253e-03,
           6.3060e-03,  5.0816e-03]]]), '../model_weights/int4/qwen3-8b-A81W41/model/rotary_emb/sin_cached.bin'), 'cos_cache': ({Tensor:(1, 4096, 128)} tensor([[[ 1.0000,  1.0000,  1.0000,  ...,  1.0000,  1.0000,  1.0000],
         [ 0.5403,  0.6925,  0.7965,  ...,  1.0000,  1.0000,  1.0000],
         [-0.4161, -0.0409,  0.2687,  ...,  1.0000,  1.0000,

## Save Qwen Decoder Layers

In [19]:
QWEN_BLOCKS = model.model.layers

In [20]:
QWEN_BLOCKS[0]

Qwen3DecoderLayer(
  (self_attn): Qwen3Attention(
    (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
    (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
    (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
    (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
    (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
    (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
  )
  (mlp): Qwen3MLP(
    (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
    (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
    (down_proj): Linear(in_features=12288, out_features=4096, bias=False)
    (act_fn): SiLU()
  )
  (input_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
  (post_attention_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
)

In [21]:
scaling = QWEN_BLOCKS[0].self_attn.scaling
print(scaling)
scaling = torch.tensor(scaling, dtype=torch.float32)
scaling


0.08838834764831845


{Tensor:()} tensor(0.0884)

In [22]:
for idx, layer in enumerate(QWEN_BLOCKS):
    fp32_state_dict = dict()
    output_dir = os.path.join(outdir_root, "model", "layers", f"layer{idx}")
    self_attn = layer.self_attn
    mlp = layer.mlp
    input_layernorm = layer.input_layernorm
    post_attention_layernorm = layer.post_attention_layernorm
    
    # save unquantized modules 
    fp32_state_dict.update(
        # self_attn
        q_norm_weight = ( self_attn.q_norm.weight, os.path.join(output_dir, "self_attn", "q_norm", "weight.bin") ),
        k_norm_weight = ( self_attn.k_norm.weight, os.path.join(output_dir, "self_attn", "k_norm", "weight.bin") ),
        # scaling factor in attention calculation
        scaling = (torch.tensor(self_attn.scaling, dtype=torch.float32), os.path.join(output_dir, "self_attn", "scaling.bin") ),
        # input_layernorm
        input_layernorm_weight = (input_layernorm.weight, os.path.join(output_dir, "input_layernorm", "weight.bin") ),
        # post_attention_layernorm
        post_attention_layernorm_weight = (post_attention_layernorm.weight, os.path.join(output_dir, "post_attention_layernorm", "weight.bin") ),

    )
    for _, path in fp32_state_dict.values():
        dirname = os.path.dirname(path)
        os.makedirs(dirname, exist_ok=True)
    with torch.no_grad():
        for name, (data, path) in fp32_state_dict.items():
            with open(path, "wb") as f:
                f.write(data.cpu().float().numpy().tobytes())
                print(f"\033[92mSaved {name} to {path}\033[0m")  # Green colored output
    
    # save quantized modules 
    quanzation_state_dict = {
      os.path.join ( output_dir, "self_attn", "q_proj"   ) : self_attn.q_proj.weight, 
      os.path.join ( output_dir, "self_attn", "k_proj"   ) : self_attn.k_proj.weight, 
      os.path.join ( output_dir, "self_attn", "v_proj"   ) : self_attn.v_proj.weight, 
      os.path.join ( output_dir, "self_attn", "qkv_proj"   ) : torch.cat([self_attn.q_proj.weight, self_attn.k_proj.weight, self_attn.v_proj.weight], dim=0), 
      os.path.join ( output_dir, "self_attn", "o_proj"   ) : self_attn.o_proj.weight, 
      os.path.join ( output_dir, "mlp"      , "gate_proj") : mlp.gate_proj.weight   , 
      os.path.join ( output_dir, "mlp"      , "up_proj"  ) : mlp.up_proj.weight     , 
      os.path.join ( output_dir, "mlp"      , "gate_up_proj"  ) : torch.cat([mlp.gate_proj.weight, mlp.up_proj.weight], dim=0), 
      os.path.join ( output_dir, "mlp"      , "down_proj") : mlp.down_proj.weight   , 
    }
    quantize_and_save(quanzation_state_dict)

Saved q_norm_weight to ../model_weights/int4/qwen3-8b-A81W41/model/layers/layer0/self_attn/q_norm/weight.bin
Saved k_norm_weight to ../model_weights/int4/qwen3-8b-A81W41/model/layers/layer0/self_attn/k_norm/weight.bin
Saved scaling to ../model_weights/int4/qwen3-8b-A81W41/model/layers/layer0/self_attn/scaling.bin
Saved input_layernorm_weight to ../model_weights/int4/qwen3-8b-A81W41/model/layers/layer0/input_layernorm/weight.bin
Saved post_attention_layernorm_weight to ../model_weights/int4/qwen3-8b-A81W41/model/layers/layer0/post_attention_layernorm/weight.bin
Max error: 0.027344, mean error: 0.000000, relative error: 102.166855%
Saved ../model_weights/int4/qwen3-8b-A81W41/model/layers/layer0/self_attn/q_proj/weight_int4.bin
Saved ../model_weights/int4/qwen3-8b-A81W41/model/layers/layer0/self_attn/q_proj/scaling_factor_int4.bin
Saved ../model_weights/int4/qwen3-8b-A81W41/model/layers/layer0/self_attn/q_proj/offset_int4.bin
Saved ../model_weights/int4/qwen3-8b-A81W41/model/layers/layer0

In [23]:
q_proj_weight = QWEN_BLOCKS[0].self_attn.q_proj.weight.detach().cpu()

In [24]:
qs, d, m, zp = quantize_method(q_proj_weight, False)

Max error: 0.027344, mean error: 0.000000, relative error: 102.166855%


In [25]:
d

array([0.00292969, 0.00262858, 0.00414225, ..., 0.00810547, 0.00784505,
       0.00564779], shape=(524288,), dtype=float32)

In [26]:
qs

array([[ 23, 152,  87, ...,  40, 245, 196],
       [169, 102, 182, ...,  90, 204, 138],
       [ 86, 242,  57, ..., 214, 196,  85],
       ...,
       [137,  97, 191, ..., 204, 105, 167],
       [ 84, 181, 100, ...,  67, 107, 222],
       [162,  74, 221, ...,  34, 183, 200]],
      shape=(262144, 32), dtype=uint8)

In [27]:
m

array([-0.02001953, -0.02160645, -0.03442383, ..., -0.05712891,
       -0.05639648, -0.04248047], shape=(524288,), dtype=float32)

In [28]:
zp

array([0, 0, 0, ..., 0, 0, 0], shape=(524288,), dtype=int8)

In [29]:
q_proj_weight = QWEN_BLOCKS[0].self_attn.q_proj.weight.detach().cpu()
k_proj_weight = QWEN_BLOCKS[0].self_attn.k_proj.weight.detach().cpu()
v_proj_weight = QWEN_BLOCKS[0].self_attn.v_proj.weight.detach().cpu()
q_proj_weight.shape, k_proj_weight.shape, v_proj_weight.shape

(torch.Size([4096, 4096]), torch.Size([1024, 4096]), torch.Size([1024, 4096]))

In [30]:
qkv_proj_weight = torch.cat([q_proj_weight, k_proj_weight, v_proj_weight], dim=0)
qkv_proj_weight.shape

torch.Size([6144, 4096])

In [31]:
qkv_proj_weight[5120]

{Tensor:(4096,)} tensor([-0.0040,  0.0542, -0.0060,  ..., -0.0376,  0.0171, -0.0004])

In [32]:
v_proj_weight[0]

{Tensor:(4096,)} tensor([-0.0040,  0.0542, -0.0060,  ..., -0.0376,  0.0171, -0.0004])

In [33]:
gate_proj_weight = QWEN_BLOCKS[0].mlp.gate_proj.weight.detach().cpu()
gate_proj_weight

{Tensor:(12288, 4096)} tensor([[-0.0238,  0.0130,  0.0042,  ..., -0.0260, -0.0138, -0.0225],
        [-0.0121, -0.0131, -0.0571,  ...,  0.0173,  0.0175,  0.0200],
        [ 0.0043,  0.0342,  0.0132,  ..., -0.0361, -0.0040,  0.0115],
        ...,
        [-0.0107,  0.0134,  0.0136,  ...,  0.0330,  0.0153,  0.0330],
        [-0.0165,  0.0184,  0.0024,  ...,  0.0081, -0.0413,  0.0579],
        [ 0.0039,  0.0171,  0.0110,  ...,  0.0317,  0.0162,  0.0021]])

In [34]:
up_proj_weight = QWEN_BLOCKS[0].mlp.up_proj.weight.detach().cpu()
up_proj_weight

{Tensor:(12288, 4096)} tensor([[-0.0168,  0.0153, -0.0432,  ...,  0.0152, -0.0464, -0.0061],
        [-0.0019,  0.0304,  0.0303,  ..., -0.0459, -0.0025, -0.0205],
        [ 0.0069,  0.0277, -0.0112,  ..., -0.0249, -0.0139, -0.0457],
        ...,
        [ 0.0391, -0.0123, -0.0147,  ..., -0.0537,  0.0137, -0.0022],
        [ 0.0245, -0.0223, -0.0366,  ..., -0.0049, -0.0005, -0.0066],
        [-0.0131,  0.0015,  0.0083,  ..., -0.0026, -0.0021, -0.0344]])